# Ders 7: Difüzyon ve Skor Tabanlı Üretici Modeller

**İleri Derin Öğrenme** — Haydar Kılıç

Ön koşul: *Derin Öğrenme*, Ders 8 (VAE, GAN, difüzyona giriş) ve Ders 12 (Normalleştirici akışlar).

Giriş dersinde difüzyonu "gürültü ekle, gidermeyi öğren" olarak sunmuştuk. Bu defterde asıl teoriyi
kuruyoruz: gürültü giderme işleminin **skor kestirimi** oluşu, ters zamanlı SDE ve onun
deterministik ikizi olan olasılık-akış ODE'si, DDIM, sınıflandırıcısız yönlendirme ve gürültü
programının gerçekte neyi seçtiği.

Bir kolaylık bunların hepsini tam olarak hesaplanabilir kılıyor: bir Gauss karışımı için
$\nabla_x \log p_t(x)$ skorunun her gürültü seviyesinde **kapalı formu** vardır. Böylece hiçbir ağ
eğitmeden gerçek örnekleyiciyi çalıştırabilir ve algoritmaları optimizasyon hatasından yalıtılmış
biçimde görebiliriz.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams["figure.dpi"] = 100
print("Kütüphaneler yüklendi.")


## 1. İleri Süreç ve Kapalı Formu

Varyans-koruyan ileri süreç adım adım Gauss gürültüsü ekler:

$$q(x_t \mid x_{t-1}) = \mathcal{N}\!\left(\sqrt{1-\beta_t}\,x_{t-1},\ \beta_t I\right),$$

ve — eğitimi mümkün kılan özellik — tek adımda atlanabilir. $\alpha_t = 1-\beta_t$ ve
$\bar\alpha_t = \prod_{s\le t}\alpha_s$ ile:

$$q(x_t \mid x_0) = \mathcal{N}\!\left(\sqrt{\bar\alpha_t}\,x_0,\ (1-\bar\alpha_t) I\right)
\quad\Longleftrightarrow\quad x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon .$$

Eğitim sırasında hiçbir benzetim gerekmez: $t$ örnekle, $\epsilon$ örnekle, doğrudan oraya atla. Bir
gürültü seviyesinin en yararlı özeti **sinyal-gürültü oranıdır**:
$\text{SNR}(t) = \bar\alpha_t/(1-\bar\alpha_t)$.


In [ ]:
T = 1000
def schedule(kind="kosinüs", T=T, s=0.008):
    t = np.arange(T+1)/T
    if kind == "doğrusal":
        betas = np.linspace(1e-4, 0.02, T)
        return np.concatenate([[1.0], np.cumprod(1-betas)])
    f = np.cos((t + s)/(1 + s) * np.pi/2)**2
    return f/f[0]

ab_lin, ab_cos = schedule("doğrusal"), schedule("kosinüs")
snr = lambda ab: ab/(1-ab+1e-12)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].plot(ab_lin, lw=2, label="doğrusal")
axes[0].plot(ab_cos, lw=2, label="kosinüs")
axes[0].set_xlabel("t"); axes[0].set_ylabel(r"$\bar\alpha_t$")
axes[0].set_title("Korunan sinyal"); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

axes[1].semilogy(snr(ab_lin)[1:], lw=2, label="doğrusal")
axes[1].semilogy(snr(ab_cos)[1:], lw=2, label="kosinüs")
axes[1].axhline(1, ls="--", c="k", lw=1)
axes[1].set_xlabel("t"); axes[1].set_ylabel("SNR (log)")
axes[1].set_title("Doğrusal program bilgiyi çok erken yok ediyor")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3, which="both")

# Veri yörünge boyunca nasıl görünüyor
x0 = np.array([1.5])
ts = [0, 100, 300, 600, 900, 1000]
xs = np.linspace(-3, 3, 300)
for t_ in ts:
    a = ab_cos[t_]
    axes[2].plot(xs, np.exp(-(xs-np.sqrt(a)*x0[0])**2/(2*(1-a)+1e-9))/np.sqrt(2*np.pi*(1-a)+1e-9),
                 lw=1.8, label=f"t={t_}")
axes[2].set_xlabel("x"); axes[2].set_ylabel("q(x_t | x_0)")
axes[2].set_title("Tek bir veri noktasının N(0, I)'ya difüzyonu"); axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()
print("Kosinüs programın SNR > 1'de geçen kısmı:", f"{(snr(ab_cos) > 1).mean():.2f}")
print("Doğrusal programın SNR > 1'de geçen kısmı:", f"{(snr(ab_lin) > 1).mean():.2f}")


## 2. Gürültü Gidermek Skor Kestirmektir

Ağı $\epsilon_\theta(x_t, t)$, yani eklenen gürültüyü tahmin edecek şekilde eğitin. Skorla bağlantı
$q(x_t\mid x_0)$'ın Gauss biçiminden gelir; $\log q$'nun türevi

$$\nabla_{x_t} \log q(x_t \mid x_0) = -\frac{x_t - \sqrt{\bar\alpha_t}x_0}{1-\bar\alpha_t}
= -\frac{\epsilon}{\sqrt{1-\bar\alpha_t}},$$

olur ve $x_0$ üzerinden beklenen değer alındığında koşullu skor marjinal skora dönüşür. Dolayısıyla

$$\boxed{\ s_\theta(x_t, t) \;=\; \nabla_{x_t}\log p_t(x_t) \;=\; -\frac{\epsilon_\theta(x_t,t)}{\sqrt{1-\bar\alpha_t}}\ }$$

Gürültüyü tahmin etmek, skoru tahmin etmek ve $\hat x_0$'a gürültü gidermek aynı nesnenin üç
parametrizasyonudur. Bir Gauss karışımı $p_0 = \sum_k w_k \mathcal{N}(\mu_k, \sigma_k^2)$ için
gürültülenmiş marjinal yine bir karışımdır; bu yüzden tam skoru yazıp özdeşliği sayısal olarak
doğrulayabiliriz.


In [ ]:
# Gerçek 2B veri: Gauss karışımı
W  = np.array([0.35, 0.35, 0.30])
MU = np.array([[-1.8, -1.0], [1.8, -1.0], [0.0, 1.7]])
SD = np.array([0.30, 0.30, 0.30])

def log_pt(x, ab):                       # x: (n,2) -> gürültülenmiş karışım için log p_t(x)
    var = ab*SD[:, None]**2 + (1-ab)
    mu  = np.sqrt(ab)*MU
    d2  = ((x[:, None, :] - mu[None])**2).sum(-1)
    logs = np.log(W)[None] - d2/(2*var[:, 0][None]) - np.log(2*np.pi*var[:, 0])[None]
    m = logs.max(1, keepdims=True)
    return (m[:, 0] + np.log(np.exp(logs-m).sum(1)))

def score(x, ab):                        # tam grad_x log p_t(x)
    var = (ab*SD**2 + (1-ab))
    mu  = np.sqrt(ab)*MU
    d   = x[:, None, :] - mu[None]
    logw = np.log(W)[None] - (d**2).sum(-1)/(2*var[None]) - np.log(2*np.pi*var)[None]
    r = np.exp(logw - logw.max(1, keepdims=True)); r /= r.sum(1, keepdims=True)
    return (-r[..., None]*d/var[None, :, None]).sum(1)

# Skorun, log p_t'nin sonlu farkına karşı sayısal kontrolü
ab = 0.5
xq = np.random.randn(5, 2)
h  = 1e-5
num = np.stack([(log_pt(xq+h*np.eye(2)[i], ab) - log_pt(xq-h*np.eye(2)[i], ab))/(2*h) for i in range(2)], 1)
print(f"maks |analitik skor - sonlu fark| = {np.abs(score(xq, ab)-num).max():.2e}")

# Tweedie formülü: E[x0 | x_t] = (x_t + (1-abar) * skor) / sqrt(abar).
# Gerçek veri dağılımından örneklerle Monte-Carlo sonsal ortalamasına karşı doğrula.
rng_mc = np.random.default_rng(0)
x0_mc  = MU[rng_mc.choice(3, 200000, p=W)] + SD[0]*rng_mc.normal(size=(200000, 2))
tweedie = (xq + (1-ab)*score(xq, ab))/np.sqrt(ab)
mc = []
for x in xq:
    logw = -((x - np.sqrt(ab)*x0_mc)**2).sum(1)/(2*(1-ab))
    w = np.exp(logw - logw.max()); w /= w.sum()
    mc.append(w @ x0_mc)
print(f"maks |Tweedie E[x0|x_t] - Monte-Carlo kestirimi| = {np.abs(tweedie-np.array(mc)).max():.3f}")

gx, gy = np.meshgrid(np.linspace(-3.5, 3.5, 22), np.linspace(-3.5, 3.5, 22))
pts = np.stack([gx.ravel(), gy.ravel()], 1)
fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))
for ax, a in zip(axes, [0.999, 0.9, 0.5, 0.05]):
    dens = np.exp(log_pt(pts, a)).reshape(gx.shape)
    s = score(pts, a)
    ax.contourf(gx, gy, dens, 20, cmap="Blues")
    ax.quiver(gx, gy, s[:, 0].reshape(gx.shape), s[:, 1].reshape(gx.shape),
              color="crimson", alpha=0.75, width=0.004)
    ax.set_title(f"abar = {a}   (SNR = {a/(1-a):.2f})", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Gürültü azaldıkça skor alanı veriye doğru yöneliyor", fontsize=13)
plt.tight_layout(); plt.show()


## 3. Ters SDE ve Olasılık-Akış ODE'si

Anderson'ın sonucuna göre $dx = f(x,t)dt + g(t)dw$ ileri SDE'sinin bir zaman tersi vardır:

$$dx = \left[f(x,t) - g(t)^2 \nabla_x \log p_t(x)\right]dt + g(t)\,d\bar w,$$

ve bu yalnızca skora ihtiyaç duyar. Dikkat çekici olan şu: aynı $p_t$ marjinalleri tamamen
**deterministik** bir ODE tarafından da üretilir:

$$\frac{dx}{dt} = f(x,t) - \tfrac{1}{2}g(t)^2 \nabla_x \log p_t(x).$$

Bu *olasılık-akış ODE'si*, difüzyonun sürekli-normalleştirici-akış görünümüdür: tam olabilirlik
verir, deterministik ve tersinir örnekleme sağlar (böylece gizil değişkenler düzenlenebilir veya
aradeğerlenebilir) ve adım sayısını azaltmak için hazır ODE çözücüleri kullanılmasına izin verir.
Stokastik örnekleyici genellikle biraz daha çeşitli örnekler üretir; çünkü eklenen gürültü önceki
hataları düzeltebilir.


In [ ]:
abar      = np.clip(schedule("kosinüs")[1:], 1e-4, 1.0)      # abar_1 ... abar_T, indeks 0..T-1
abar_prev = np.concatenate([[1.0], abar[:-1]])
betas     = np.clip(1 - abar/abar_prev, 1e-8, 0.999)
post_var  = betas*(1-abar_prev)/(1-abar)                     # gerçek sonsal varyans
print(f"abar aralığı {abar[0]:.4f} (t=0) -> {abar[-1]:.4f} (t=T);  beta aralığı "
      f"{betas.min():.5f} -> {betas.max():.5f}")

def sample_sde(n=1500, seed=0):
    rng = np.random.default_rng(seed)
    x = rng.normal(size=(n, 2)); traj = [x.copy()]
    for t in range(T-1, -1, -1):
        s = score(x, abar[t])
        mean = (x + betas[t]*s)/np.sqrt(1-betas[t])
        x = mean + (np.sqrt(post_var[t])*rng.normal(size=x.shape) if t > 0 else 0)
        if t % 100 == 0: traj.append(x.copy())
    return x, traj

def sample_ode(n=1500, steps=50, seed=0):
    rng = np.random.default_rng(seed)
    x = rng.normal(size=(n, 2))
    grid = np.linspace(T-1, 0, steps).astype(int)
    for i in range(len(grid)-1):
        t, t_next = grid[i], grid[i+1]
        a, a_next = abar[t], abar[t_next]
        eps = -np.sqrt(1-a)*score(x, a)                     # DDIM güncellemesi (eta = 0)
        x0  = (x - np.sqrt(1-a)*eps)/np.sqrt(a)
        x   = np.sqrt(a_next)*x0 + np.sqrt(1-a_next)*eps
    return x

x_sde, traj = sample_sde()
x_ddim50 = sample_ode(steps=50)

fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))
real = MU[np.random.choice(3, 1500, p=W)] + SD[0]*np.random.randn(1500, 2)
for ax, (name, pts_) in zip(axes, [("gerçek dağılım", real), ("ata SDE, 1000 adım", x_sde),
                                   ("olasılık-akış ODE (DDIM), 50 adım", x_ddim50),
                                   ("DDIM, 5 adım", sample_ode(steps=5))]):
    ax.scatter(pts_[:, 0], pts_[:, 1], s=4, alpha=0.4)
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_title(name, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

def mode_weights(pts_):
    d = ((pts_[:, None, :]-MU[None])**2).sum(-1)
    return np.bincount(d.argmin(1), minlength=3)/len(pts_)
for name, pts_ in [("gerçek", real), ("SDE 1000", x_sde), ("DDIM 50", x_ddim50), ("DDIM 5", sample_ode(steps=5))]:
    print(f"{name:14s} mod ağırlıkları {mode_weights(pts_).round(3)}   (hedef {W})")


## 4. Adım Sayısı: DDIM Dağıtımı Neden Değiştirdi?

DDPM'in ata (ancestral) örnekleyicisi eğitim ızgarasına bağlıdır: 1000 eğitim adımı 1000 örnekleme
adımı demektir. DDIM'in güncellemesi

$$x_{t-1} = \sqrt{\bar\alpha_{t-1}}\,\hat x_0 + \sqrt{1-\bar\alpha_{t-1}-\sigma_t^2}\,\epsilon_\theta + \sigma_t z$$

ise zaman adımlarının **herhangi** bir alt dizisi için geçerli bir örnekleyicidir ve $\sigma_t = 0$
iken deterministiktir. Böylece aynı eğitilmiş ağ 1000 yerine 20–50 adımla örneklenebilir. $\eta$
parametresi ara değerleri verir: $\eta=0$ ODE'dir, $\eta=1$ DDPM'i geri getirir.


In [ ]:
def sample_ddim(n=1200, steps=50, eta=0.0, seed=1, x_init=None, noise_seed=None):
    rng = np.random.default_rng(seed if noise_seed is None else noise_seed)
    x = np.random.default_rng(seed).normal(size=(n, 2)) if x_init is None else x_init.copy()
    grid = np.linspace(T-1, 0, steps).astype(int)
    for i in range(len(grid)-1):
        t, tn = grid[i], grid[i+1]
        a, an = abar[t], abar[tn]
        eps = -np.sqrt(1-a)*score(x, a)
        x0  = (x - np.sqrt(1-a)*eps)/np.sqrt(a)
        sig = eta*np.sqrt((1-an)/(1-a))*np.sqrt(1-a/an)
        x   = np.sqrt(an)*x0 + np.sqrt(max(1-an-sig**2, 0))*eps + sig*rng.normal(size=x.shape)
    return x

def w2_proxy(pts_):                       # basit kalite göstergesi: gerçek karışıma uzaklık
    d = np.sqrt(((pts_[:, None, :]-MU[None])**2).sum(-1)).min(1)
    return d.mean()

step_list = [2, 3, 5, 10, 20, 50, 100, 250]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for eta, c in [(0.0, "steelblue"), (0.5, "seagreen"), (1.0, "crimson")]:
    q = [w2_proxy(sample_ddim(steps=s, eta=eta)) for s in step_list]
    axes[0].semilogx(step_list, q, "o-", lw=2, c=c, label=f"eta={eta}")
axes[0].set_xlabel("örnekleme adımı"); axes[0].set_ylabel("en yakın moda ortalama uzaklık")
axes[0].set_title("Daha az adım, deterministik örnekleyici"); axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3, which="both")

for ax, s in zip(axes[1:], [5, 50]):
    p_ = sample_ddim(steps=s, eta=0.0)
    ax.scatter(p_[:, 0], p_[:, 1], s=4, alpha=0.4)
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
    ax.set_title(f"DDIM eta=0, {s} adım"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

# Determinizm: başlangıç gizil değişkenini sabitle, yalnızca örnekleme gürültüsünü değiştir
z0 = np.random.default_rng(5).normal(size=(6, 2))
a1 = sample_ddim(steps=30, eta=0.0, x_init=z0, noise_seed=11)
a2 = sample_ddim(steps=30, eta=0.0, x_init=z0, noise_seed=99)
b1 = sample_ddim(steps=30, eta=1.0, x_init=z0, noise_seed=11)
b2 = sample_ddim(steps=30, eta=1.0, x_init=z0, noise_seed=99)
print(f"eta=0 : aynı gizil değişken, farklı gürültü -> çıktı farkı {np.abs(a1-a2).max():.2e}  (deterministik)")
print(f"eta=1 : aynı gizil değişken, farklı gürültü -> çıktı farkı {np.abs(b1-b2).max():.2e}  (stokastik)")


## 5. Sınıflandırıcısız Yönlendirme (CFG)

Koşullu üretim $\nabla_x \log p_t(x \mid c)$'yi gerektirir. Bayes kuralı bunu ikiye ayırır:

$$\nabla_x \log p_t(x\mid c) = \nabla_x \log p_t(x) + \nabla_x \log p_t(c \mid x).$$

**Sınıflandırıcı yönlendirmesi** ikinci terim için ayrı bir gürültülü sınıflandırıcı eğitir.
**Sınıflandırıcısız yönlendirme** bundan kaçınır: tek bir ağ, koşul rastgele düşürülerek (örneğin
zamanın %10'unda) eğitilir; böylece hem $\epsilon_\theta(x,c)$ hem $\epsilon_\theta(x,\varnothing)$
öğrenilir ve örnekleme sırasında dışarı doğru çıkarım yapılır:

$$\tilde\epsilon = \epsilon_\theta(x,\varnothing) + w\,\big(\epsilon_\theta(x,c) - \epsilon_\theta(x,\varnothing)\big).$$

$w=1$ sıradan koşullu modeldir. $w>1$ örnekleri, koşulun ortalamadan *daha olası* olduğu bölgelere
iter — daha keskin, isteme daha sadık ve ölçülebilir biçimde daha az çeşitli. Bu takas ayarlanarak
yok edilecek bir kusur değildir; mekanizmanın kendisidir.


In [ ]:
# Koşul c = "hangi mod": koşullu skor tek bileşeni tutar, koşulsuz olan üçünü de tutar
def score_cond(x, ab, k=None):
    var = (ab*SD**2 + (1-ab))
    mu  = np.sqrt(ab)*MU
    w   = W.copy()
    if k is not None:
        w = np.zeros(3); w[k] = 1.0
    d = x[:, None, :] - mu[None]
    logw = np.log(w + 1e-30)[None] - (d**2).sum(-1)/(2*var[None]) - np.log(2*np.pi*var)[None]
    r = np.exp(logw - logw.max(1, keepdims=True)); r /= r.sum(1, keepdims=True)
    return (-r[..., None]*d/var[None, :, None]).sum(1)

def sample_cfg(k, w_guide, n=1200, steps=60, seed=2):
    rng = np.random.default_rng(seed)
    x = rng.normal(size=(n, 2))
    grid = np.linspace(T-1, 0, steps).astype(int)
    for i in range(len(grid)-1):
        t, tn = grid[i], grid[i+1]
        a, an = abar[t], abar[tn]
        e_u = -np.sqrt(1-a)*score_cond(x, a, None)
        e_c = -np.sqrt(1-a)*score_cond(x, a, k)
        eps = e_u + w_guide*(e_c - e_u)
        x0  = (x - np.sqrt(1-a)*eps)/np.sqrt(a)
        x   = np.sqrt(an)*x0 + np.sqrt(1-an)*eps
    return x

ws = [0.0, 1.0, 3.0, 8.0]
fig, axes = plt.subplots(1, 5, figsize=(18, 3.9))
spreads = []
for ax, w_ in zip(axes[:4], ws):
    p_ = sample_cfg(2, w_)
    spreads.append(p_.std(0).mean())
    ax.scatter(p_[:, 0], p_[:, 1], s=4, alpha=0.4)
    ax.scatter(*MU[2], marker="*", s=180, c="crimson", zorder=4)
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"w = {w_}", fontsize=11)
axes[4].plot(ws, spreads, "o-", lw=2)
axes[4].set_xlabel("yönlendirme ağırlığı w"); axes[4].set_ylabel("örnek yayılımı (std)")
axes[4].set_title("Yönlendirme sadakati çeşitlilikle satın alır"); axes[4].grid(alpha=0.3)
plt.suptitle("Üstteki moda doğru sınıflandırıcısız yönlendirme (kırmızı yıldız)", fontsize=13)
plt.tight_layout(); plt.show()

for w_, s in zip(ws, spreads):
    print(f"w={w_:4.1f}  örnek yayılımı {s:.3f}")


## 6. Parametrizasyonlar ve Kayıp Ağırlıklandırması

Üç eşdeğer hedef ve bunu önemsemek için pratik bir neden:

| Tahmin | Formül | İyi davrandığı bölge |
|---|---|---|
| $\epsilon$-tahmini | $\epsilon_\theta(x_t,t)$ | yüksek gürültü (standart seçim) |
| $x_0$-tahmini | $\hat x_0 = (x_t - \sqrt{1-\bar\alpha_t}\epsilon)/\sqrt{\bar\alpha_t}$ | düşük gürültü |
| $v$-tahmini | $v = \sqrt{\bar\alpha_t}\epsilon - \sqrt{1-\bar\alpha_t}x_0$ | her yerde; damıtma ve yüksek çözünürlükte kullanılır |

Gerçek varyasyonel sınır, zaman adımı başına kayıpları $\propto 1/\text{SNR}$ ile ağırlıklandırır;
ama DDPM'in *basitleştirilmiş* amacı bu ağırlığı atar ve düz bir
$\lVert \epsilon - \epsilon_\theta\rVert^2$ kullanır. Bu yalnızca kolaylık için yapılmış bir
basitleştirme değildir — tekdüze ağırlıklandırma orta gürültü seviyelerini öne çıkarır ve algısal
olarak önemli yapı orada belirlenir; basitleştirilmiş kaybın tam ELBO'dan daha iyi örnekler
üretmesinin büyük kısmı buradan gelir.


In [ ]:
ab_c = schedule("kosinüs")[1:]
snr_c = ab_c/(1-ab_c)
t_ax = np.arange(len(ab_c))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].semilogy(t_ax, 1/snr_c, lw=2, label="ELBO ağırlığı  ~ 1/SNR")
axes[0].semilogy(t_ax, np.ones_like(t_ax), lw=2, label="basitleştirilmiş kayıp (tekdüze)")
axes[0].set_xlabel("t"); axes[0].set_ylabel("adım başına kayıp ağırlığı (log)")
axes[0].set_title("İki amaç fonksiyonu neyi öne çıkarıyor"); axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3, which="both")

# Hata büyütmesi: eps'teki sabit bir hata x0'da farklı bir hataya dönüşür
err = 0.1
amp_eps = np.sqrt(1-ab_c)/np.sqrt(ab_c)
axes[1].semilogy(t_ax, amp_eps*err, lw=2, label="eps-tahmini -> x0'daki hata")
axes[1].semilogy(t_ax, np.full_like(t_ax, err, dtype=float), lw=2, label="x0-tahmini -> x0'daki hata")
axes[1].set_xlabel("t"); axes[1].set_ylabel("x0'da ortaya çıkan hata (log)")
axes[1].set_title("eps-tahmini çok düşük gürültüde neden zorlanır")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3, which="both")

v_x0  = -np.sqrt(1-ab_c)
v_eps =  np.sqrt(ab_c)
axes[2].plot(t_ax, v_eps, lw=2, label="v içinde eps ağırlığı")
axes[2].plot(t_ax, -v_x0, lw=2, label="v içinde x0 ağırlığı")
axes[2].set_xlabel("t"); axes[2].set_title("v-tahmini iki hedefi harmanlar")
axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print("t -> 0 iken eps-parametrizasyonu sqrt(abar) ~ 1'e böler; t -> T iken ise kararlı olan odur.")


## 7. Difüzyon, Üretici Modeller Arasında Nerede Duruyor?

| Model | Eğitim | Örnekleme | Tam olabilirlik | Tipik başarısızlık |
|---|---|---|---|---|
| GAN | çekişmeli, kararsız | 1 adım | hayır | mod çökmesi |
| VAE | kararlı ELBO | 1 adım | alt sınır | bulanık örnekler |
| Normalleştirici akış | tam MLE | 1 adım | **evet** | mimari tersinir olmak zorunda |
| Otoregresif | kararlı MLE | $n$ adım | **evet** | yavaş örnekleme, sabit sıralama |
| **Difüzyon** | kararlı regresyon | çok adım | PF-ODE üzerinden | yavaş örnekleme (DDIM / damıtma ile hafifler) |

Difüzyonun avantajı üretimi bir **regresyon** problemine — kararlı bir gürültü giderme kaybına —
çevirmesidir; bedeli ise yinelemeli örneklemedir. O günden bu yana yapılan araştırmanın çoğu bu
bedeli geri satın almakla ilgilidir: DDIM, yüksek mertebeli çözücüler, tutarlılık (consistency) ve
kademeli damıtma ve tüm süreci sıkıştırılmış bir otokodlayıcı uzayında çalıştıran gizil (latent)
difüzyon.

## 8. Özet

| Kavram | Açıklama |
|---|---|
| **Kapalı formlu ileri süreç** | $x_t = \sqrt{\bar\alpha_t}x_0 + \sqrt{1-\bar\alpha_t}\epsilon$; eğitimde benzetim yok |
| **SNR** | $\bar\alpha_t/(1-\bar\alpha_t)$; bir gürültü programının dürüst tarifi |
| **Kosinüs program** | Yörüngenin daha çoğunu yararlı SNR'de geçirir |
| **Skor özdeşliği** | $s_\theta = -\epsilon_\theta/\sqrt{1-\bar\alpha_t}$ |
| **Ters SDE** | Zaman tersi yalnızca skora ihtiyaç duyar |
| **Olasılık-akış ODE** | Deterministik, tersinir, tam olabilirlik, az adımlı örnekleme |
| **DDIM** | Herhangi bir zaman adımı alt dizisinde geçerli; $\eta$ ODE ↔ DDPM arasında geçiş yapar |
| **Sınıflandırıcısız yönlendirme** | $\epsilon_u + w(\epsilon_c-\epsilon_u)$; sadakat, çeşitlilikle takas edilir |
| **Parametrizasyonlar** | $\epsilon$, $x_0$, $v$ — teoride eşdeğer, sayısal koşullanmada farklı |
| **Basitleştirilmiş kayıp** | Tekdüze ağırlık orta gürültüyü öne çıkarır; tam ELBO'dan daha iyi örnekler |

**Sonraki Defter →** İleri Pekiştirmeli Öğrenme ve RLHF
